See how many areas/parks have been completed

In [674]:
import pandas as pd
import geopandas as gpd
import os

In [675]:
output_dir = "../workflow_outputs/"
filenames_file = "england_filenames.csv"
park_id_file = "all_parks_ids.csv"

In [676]:
filenames = pd.read_csv(filenames_file)
park_ids = pd.read_csv(park_id_file)

In [677]:
# want a subset of the parks that are in process, so we can check the progress of the workflow
# first_park_index = 29
# last_park_index = first_park_index + 30
# filenames= filenames.iloc[first_park_index:last_park_index]

In [678]:
filenames

,filename
0,Hartlepool_pp_or_g_cmb.geojson
1,Middlesbrough_pp_or_g_cmb.geojson
2,Redcar and Cleveland_pp_or_g_cmb.geojson
3,Stockton-on-Tees_pp_or_g_cmb.geojson
4,Darlington_pp_or_g_cmb.geojson
...,...
291,Trafford_pp_or_g_cmb.geojson
292,Wigan_pp_or_g_cmb.geojson
293,Knowsley_pp_or_g_cmb.geojson
294,Liverpool_pp_or_g_cmb.geojson


In [679]:
# for each filename in filenames
# get foldername by removing the .geojson
# check if the folder exists in output_dir
# add existing folder to a list

in_process_areas = []
for filename in filenames["filename"]:
    foldername = filename.replace(".geojson", "")
    folderpath = output_dir + foldername
    if os.path.exists(folderpath):
        in_process_areas.append(foldername)

In [680]:
100*len(in_process_areas)/len(filenames)

48.648648648648646

In [681]:
in_process_area_names = []
for filename in in_process_areas:
    areaname = filename.replace("_pp_or_g_cmb", "")
    in_process_area_names.append(areaname)

In [682]:
area_df = pd.DataFrame({"area_name": in_process_area_names,
                        "filename": in_process_areas,
                        "folder_path": [output_dir + filename for filename in in_process_areas]})

In [683]:
park_ids

,country,region_id,authority_id,auth_name_e,old_park_id,new_park_id
0,Wales,W10000009,W06000011,Swansea,SWANS_841e2e988f55;SWANS_9add957b9b60,3816d9f14391
1,Wales,W10000009,W06000011,Swansea,SWANS_64180ad89ee2;SWANS_7875d4152bbb,eeb8a45abba6
2,Wales,W10000009,W06000011,Swansea,SWANS_0077e0499336;SWANS_003142c8af8d,2afa4425d6cc
3,Wales,W10000009,W06000011,Swansea,SWANS_8c5a5b68c7e1;SWANS_fb7bfbb4dae0,ee93e1ee473b
4,Wales,W10000009,W06000011,Swansea,SWANS_8119996c470b,a7f47d7bf887
...,...,...,...,...,...,...
217614,England,E12000002,E08000013,St. Helens,STHEL_6a530cb0f057,e97074068e56
217615,England,E12000002,E08000013,St. Helens,STHEL_2df5277ef0c9,f9627d5e096b
217616,England,E12000002,E08000013,St. Helens,STHEL_04f0db6f2403,7957ab4ecde8
217617,England,E12000002,E08000013,St. Helens,STHEL_0384f41414f0,42c0619d2802


In [684]:
area_df

,area_name,filename,folder_path
0,Hartlepool,Hartlepool_pp_or_g_cmb,../workflow_outputs/Hartlepool_pp_or_g_cmb
1,Middlesbrough,Middlesbrough_pp_or_g_cmb,../workflow_outputs/Middlesbrough_pp_or_g_cmb
2,Redcar and Cleveland,Redcar and Cleveland_pp_or_g_cmb,../workflow_outputs/Redcar and Cleveland_pp_or...
3,Stockton-on-Tees,Stockton-on-Tees_pp_or_g_cmb,../workflow_outputs/Stockton-on-Tees_pp_or_g_cmb
4,Darlington,Darlington_pp_or_g_cmb,../workflow_outputs/Darlington_pp_or_g_cmb
...,...,...,...
139,Hyndburn,Hyndburn_pp_or_g_cmb,../workflow_outputs/Hyndburn_pp_or_g_cmb
140,Lancaster,Lancaster_pp_or_g_cmb,../workflow_outputs/Lancaster_pp_or_g_cmb
141,Pendle,Pendle_pp_or_g_cmb,../workflow_outputs/Pendle_pp_or_g_cmb
142,Preston,Preston_pp_or_g_cmb,../workflow_outputs/Preston_pp_or_g_cmb


In [685]:
# for each folder_path in the area_df, find all parks in the area
# in the park_ids dataframe, find all parks with area_name in the area_df
# add the total number of parks in the area to the area_df
area_df["num_parks_total"] = area_df["area_name"].apply(lambda x: len(park_ids[park_ids["auth_name_e"] == x]))
area_df["expected_park_files"] = area_df["num_parks_total"].apply(lambda x: x * 2)

In [686]:
# then count the number of parks in the associated folder_path and add that to the area_df
def count_files_in_folder(folder_path):
    park_files = [f for f in os.listdir(folder_path) if f.endswith(".geojson")]
    return len(park_files)

area_df["completed_files"] = area_df["folder_path"].apply(count_files_in_folder)
area_df["completed_parks"] = area_df["completed_files"] / 2

In [687]:
area_df

,area_name,filename,folder_path,num_parks_total,expected_park_files,completed_files,completed_parks
0,Hartlepool,Hartlepool_pp_or_g_cmb,../workflow_outputs/Hartlepool_pp_or_g_cmb,38,76,76,38.0
1,Middlesbrough,Middlesbrough_pp_or_g_cmb,../workflow_outputs/Middlesbrough_pp_or_g_cmb,75,150,150,75.0
2,Redcar and Cleveland,Redcar and Cleveland_pp_or_g_cmb,../workflow_outputs/Redcar and Cleveland_pp_or...,75,150,150,75.0
3,Stockton-on-Tees,Stockton-on-Tees_pp_or_g_cmb,../workflow_outputs/Stockton-on-Tees_pp_or_g_cmb,393,786,786,393.0
4,Darlington,Darlington_pp_or_g_cmb,../workflow_outputs/Darlington_pp_or_g_cmb,65,130,130,65.0
...,...,...,...,...,...,...,...
139,Hyndburn,Hyndburn_pp_or_g_cmb,../workflow_outputs/Hyndburn_pp_or_g_cmb,69,138,138,69.0
140,Lancaster,Lancaster_pp_or_g_cmb,../workflow_outputs/Lancaster_pp_or_g_cmb,177,354,354,177.0
141,Pendle,Pendle_pp_or_g_cmb,../workflow_outputs/Pendle_pp_or_g_cmb,73,146,112,56.0
142,Preston,Preston_pp_or_g_cmb,../workflow_outputs/Preston_pp_or_g_cmb,121,242,6,3.0


In [688]:
area_df["pct_complete"] = 100* area_df["completed_files"]/area_df["expected_park_files"]

In [689]:
area_df.sort_values("pct_complete", ascending=True)

,area_name,filename,folder_path,num_parks_total,expected_park_files,completed_files,completed_parks,pct_complete
49,Isles of Scilly,Isles of Scilly_pp_or_g_cmb,../workflow_outputs/Isles of Scilly_pp_or_g_cmb,12,24,0,0.0,0.000000
142,Preston,Preston_pp_or_g_cmb,../workflow_outputs/Preston_pp_or_g_cmb,121,242,6,3.0,2.479339
61,North Yorkshire,North Yorkshire_pp_or_g_cmb,../workflow_outputs/North Yorkshire_pp_or_g_cmb,1049,2098,60,30.0,2.859867
82,Torridge,Torridge_pp_or_g_cmb,../workflow_outputs/Torridge_pp_or_g_cmb,519,1038,36,18.0,3.468208
137,Chorley,Chorley_pp_or_g_cmb,../workflow_outputs/Chorley_pp_or_g_cmb,623,1246,63,31.5,5.056180
...,...,...,...,...,...,...,...,...
48,Cornwall,Cornwall_pp_or_g_cmb,../workflow_outputs/Cornwall_pp_or_g_cmb,1351,2702,2702,1351.0,100.000000
50,Wiltshire,Wiltshire_pp_or_g_cmb,../workflow_outputs/Wiltshire_pp_or_g_cmb,832,1664,1664,832.0,100.000000
51,Bedford,Bedford_pp_or_g_cmb,../workflow_outputs/Bedford_pp_or_g_cmb,236,472,472,236.0,100.000000
44,County Durham,County Durham_pp_or_g_cmb,../workflow_outputs/County Durham_pp_or_g_cmb,1026,2052,2052,1026.0,100.000000
